# Sampling With The RR Protein Language Model And PF00196 Classifier

This notebook is the compact sampling workflow:

1. Load the RR sDO checkpoint.
2. Load the trained PF00196 prefix classifier.
3. Generate unsteered AR samples or classifier-guided ILMC/FUDGE samples.
4. Save generated sequences as FASTA.

Run `Prefix_Classifier/classifier_training.ipynb` first if `checkpoints/Prefix_Classifier/pf00196_prefix_string_classifier.pt` is missing.

In [1]:
from pathlib import Path
from contextlib import nullcontext
import itertools
import os
import sys

import numpy as np
import torch
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Paths And Sampling Configuration

Edit this cell for the run you want. `METHOD` can be `"AR"`, `"ILMC"`, or `"FUDGE"`.

In [2]:
PLM_CHECKPOINT = Path("../checkpoints/RR_sDO/RR_sDO_iter17999.pt")
CLASSIFIER_CHECKPOINT = Path("../checkpoints/Prefix_Classifier/pf00196_prefix_string_classifier.pt")

METHOD = "ILMC"
OUTPUT_DIR = Path("generated_samples")
N = 1024
MAX_CONTEXT = 112
BETA_SAMPLING = 1.8
LAMBDA_GUIDANCE = 5

NUM_ITERATIONS = 56
NUM_NEW_TOKENS = 2
K_RESAMPLE = 10
NUM_RESAMPLING_STEPS = 10

if METHOD == "FUDGE":
    NUM_ITERATIONS = 112
    K_RESAMPLE = 1
    NUM_NEW_TOKENS = 1
    NUM_RESAMPLING_STEPS = 20

## Imports From The Language-Model Code

The sampling helpers are bundled locally in `Sampling/sampling_utils/`.

In [3]:
SAMPLING_UTILS_DIR = Path("sampling_utils")
sys.path.insert(0, str(SAMPLING_UTILS_DIR.resolve()))

missing = []
for required in ["transformer.py", "Transformer_Reint.py", "mc.py"]:
    if not (SAMPLING_UTILS_DIR / required).exists():
        missing.append(required)
if missing:
    raise FileNotFoundError(
        "Missing sampling dependency files: "
        + ", ".join(missing)
        + ". The sampling_utils folder is incomplete."
    )

from transformer import GPTTransformer
from Transformer_Reint import load_dataset
from mc import (
    continue_generation,
    generate_block,
    metropolis_acceptance,
    resample_from_last_k,
    score_sequences,
)

## Tokenizer And RR sDO Model

In [4]:
DN_PATH = Path("../Training_DATA/RR_aligned.fasta")
with DN_PATH.open("r", encoding="utf-8") as handle:
    joined_sequences = "\n".join(handle.read().splitlines()[1::2])

chars = sorted(set(joined_sequences))
PAD_SYMBOL = "?"
chars = chars + [PAD_SYMBOL]
stoi = {ch: idx for idx, ch in enumerate(chars)}
itos = {idx: ch for ch, idx in stoi.items()}
pad_token = stoi[PAD_SYMBOL]
vocab_size = len(stoi)
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[int(i)] for i in ids)

_ = load_dataset(
    seq_path=str(DN_PATH),
    label_path=".",
    block_size=MAX_CONTEXT,
    train_size=10000,
    encode_fn=encode,
    pad_token=pad_token,
)

legacy_model = GPTTransformer(
    vocab_size=vocab_size,
    embed_dim=64,
    num_layers=16,
    num_heads=16,
    mlp_ratio=4,
    dropout_p=0.1,
    pad_id=pad_token,
)
legacy_model.load_state_dict(torch.load(PLM_CHECKPOINT, map_location="cpu"))
legacy_model = legacy_model.to(device).eval()

if hasattr(torch, "compile"):
    legacy_model = torch.compile(legacy_model)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f"Loaded RR sDO from {PLM_CHECKPOINT}")

Train sequences: 10000, Validation sequences: 845395
Train positives: 10000 (100.00%), negatives: 0 (0.00%)
Val positives: 845395 (100.00%), negatives: 0 (0.00%)
Loaded RR sDO from ../checkpoints/RR_sDO/RR_sDO_iter17999.pt


/home/calvanesef/miniconda3/envs/torch_env/lib/python3.11/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


## Classifier Loading 

In [7]:
class PrefixStringClassifier(nn.Module):
    def __init__(self, vocab_size, block_size, pad_token, embed_dim=16, hidden_dim=16):
        super().__init__()
        self.block_size = block_size
        self.pad_token = pad_token
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token)
        self.classifier = nn.Sequential(
            nn.Linear(block_size * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, tokens, prefix_lengths=None):
        embedded = self.embedding(tokens)
        return self.classifier(embedded.reshape(tokens.shape[0], -1)).squeeze(1)


clf_ckpt = torch.load(CLASSIFIER_CHECKPOINT, map_location="cpu")
sequence_length = int(clf_ckpt["sequence_length"])
clf_model = PrefixStringClassifier(
    vocab_size=int(clf_ckpt["vocab_size"]),
    block_size=sequence_length,
    pad_token=int(clf_ckpt["pad_token"]),
    embed_dim=int(clf_ckpt["embed_dim"]),
    hidden_dim=int(clf_ckpt["hidden_dim"]),
)
clf_model.load_state_dict(clf_ckpt["state_dict"])
clf_model = clf_model.to(device).eval()

@torch.inference_mode()
def classifier_scores(sequences):
    prefix_tokens = sequences[:, 1:]
    prefix_lengths = (prefix_tokens != pad_token).sum(dim=1)
    if prefix_tokens.shape[1] >= sequence_length:
        clf_input = prefix_tokens[:, :sequence_length]
    else:
        padding = torch.full(
            (sequences.shape[0], sequence_length - prefix_tokens.shape[1]),
            pad_token,
            dtype=sequences.dtype,
            device=sequences.device,
        )
        clf_input = torch.cat([prefix_tokens, padding], dim=1)

    logits = clf_model(clf_input, prefix_lengths)
    return torch.nn.functional.logsigmoid(logits)


print(f"Loaded classifier from {CLASSIFIER_CHECKPOINT}")
print(f"Classifier sequence length: {sequence_length}")

Loaded classifier from ../checkpoints/Prefix_Classifier/pf00196_prefix_string_classifier.pt
Classifier sequence length: 111


## FASTA Saving Function

In [5]:
def save_fasta(path, sequences):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for i in range(sequences.size(0)):
            seq = sequences[i].detach().cpu()
            seq = seq[seq != pad_token]
            decoded = decode(seq.tolist())
            if decoded.endswith("\n"):
                decoded = decoded[:-1]
            handle.write(f">Sequence_{i}\n{decoded}\n")
    print(f"Saved {sequences.size(0):,} sequences to {path}")

## Sampling Run

For `"AR"`, the model samples the full sequence without classifier guidance. For `"ILMC"` or `"FUDGE"`, the Metropolis energy includes the classifier score.

In [8]:
@torch.inference_mode()
def run_ar(beta_sampling=BETA_SAMPLING, n=N, num_tokens=MAX_CONTEXT):
    sequences, legacy_logp, sampling_logp = generate_block(
        legacy_model,
        beta_sampling=beta_sampling,
        N=n,
        num_tokens=num_tokens,
        max_context=MAX_CONTEXT,
        device=device,
        bos_eos_token=stoi["\n"],
        pad_token=pad_token,
    )
    return sequences, legacy_logp, sampling_logp


@torch.inference_mode()
def run_guided(
    beta_sampling=BETA_SAMPLING,
    lambda_guidance=LAMBDA_GUIDANCE,
    n=N,
    num_iterations=NUM_ITERATIONS,
    num_new_tokens=NUM_NEW_TOKENS,
    k=K_RESAMPLE,
    num_resampling_steps=NUM_RESAMPLING_STEPS,
):
    sequences, legacy_logp, sampling_logp = generate_block(
        legacy_model,
        beta_sampling=beta_sampling,
        N=n,
        num_tokens=num_new_tokens,
        max_context=MAX_CONTEXT,
        device=device,
        bos_eos_token=stoi["\n"],
        pad_token=pad_token,
    )
    scores = classifier_scores(sequences)

    for iteration in range(num_iterations):
        for step in range(num_resampling_steps):
            proposal, proposal_legacy_logp, proposal_sampling_logp = resample_from_last_k(
                sequences,
                legacy_logp,
                sampling_logp,
                legacy_model,
                beta_sampling=beta_sampling,
                max_context=MAX_CONTEXT,
                device=device,
                bos_eos_token=stoi["\n"],
                pad_token=pad_token,
                k=k,
            )
            proposal_scores = classifier_scores(proposal)

            log_prob_current = legacy_logp.sum(dim=1)
            log_prob_proposal = proposal_legacy_logp.sum(dim=1)
            log_q_current = sampling_logp.sum(dim=1)
            log_q_proposal = proposal_sampling_logp.sum(dim=1)

            energy_current = -beta_sampling * log_prob_current - lambda_guidance * scores + log_q_current
            energy_proposal = -beta_sampling * log_prob_proposal - lambda_guidance * proposal_scores + log_q_proposal
            accept = metropolis_acceptance(energy_0=energy_current, energy_1=energy_proposal, device=device)

            sequences = torch.where(accept.unsqueeze(1), proposal, sequences)
            legacy_logp = torch.where(accept.unsqueeze(1), proposal_legacy_logp, legacy_logp)
            sampling_logp = torch.where(accept.unsqueeze(1), proposal_sampling_logp, sampling_logp)
            scores = torch.where(accept, proposal_scores, scores)

            if step == 0 or step == num_resampling_steps - 1:
                print(
                    f"iter={iteration:03d} step={step:02d} "
                    f"accept={accept.float().mean().item():.3f} "
                    f"classifier={scores.mean().item():.4f} "
                    f"logP_LM={log_prob_current.mean().item():.2f}"
                )

        if iteration < num_iterations - 1:
            sequences, legacy_logp, sampling_logp = continue_generation(
                sequences,
                legacy_logp,
                sampling_logp,
                legacy_model,
                beta_sampling=beta_sampling,
                num_new_tokens=num_new_tokens,
                max_context=MAX_CONTEXT,
                device=device,
                bos_eos_token=stoi["\n"],
                pad_token=pad_token,
            )
            scores = classifier_scores(sequences)

    return sequences, legacy_logp, sampling_logp


if METHOD == "AR":
    sequences, legacy_logp, sampling_logp = run_ar()
else:
    sequences, legacy_logp, sampling_logp = run_guided()

classifier_mean = classifier_scores(sequences).mean().item()
pad_counts = (sequences != pad_token).sum(dim=1).float().clamp(min=1.0)
lm_mean = (score_sequences(sequences, legacy_model, pad_token=pad_token) / pad_counts).mean().item()
output_path = OUTPUT_DIR / f"{METHOD}_beta{BETA_SAMPLING}_lambda{LAMBDA_GUIDANCE}_rawlogP{classifier_mean:.3f}_logPlm{lm_mean:.3f}.fasta"
save_fasta(output_path, sequences)
print(f"mean raw logP(PF00196): {classifier_mean:.4f}")
print(f"mean per-token LM logP: {lm_mean:.4f}")

iter=000 step=00 accept=0.770 classifier=-1.0627 logP_LM=-1.38
iter=000 step=09 accept=0.441 classifier=-0.8329 logP_LM=-1.98


W0428 20:53:23.512000 520630 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] torch._dynamo hit config.recompile_limit (8)
W0428 20:53:23.512000 520630 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8]    function: 'forward' (/home/calvanesef/Desktop/reviewer_package/Sampling/sampling_utils/transformer.py:194)
W0428 20:53:23.512000 520630 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8]    last reason: 0/7: input_ids.stride()[0] == input_ids.size()[1]  # (unknown source input_ids.stride()[0], please file a bug)
W0428 20:53:23.512000 520630 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0428 20:53:23.512000 520630 site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html


iter=001 step=00 accept=0.413 classifier=-0.7561 logP_LM=-3.87
iter=001 step=09 accept=0.237 classifier=-0.5556 logP_LM=-4.23
iter=002 step=00 accept=0.261 classifier=-0.6383 logP_LM=-4.85
iter=002 step=09 accept=0.769 classifier=-0.5413 logP_LM=-4.91
iter=003 step=00 accept=0.753 classifier=-0.5980 logP_LM=-7.44
iter=003 step=09 accept=0.102 classifier=-0.2191 logP_LM=-7.28
iter=004 step=00 accept=0.279 classifier=-0.2058 logP_LM=-9.09
iter=004 step=09 accept=0.976 classifier=-0.1060 logP_LM=-8.78
iter=005 step=00 accept=0.479 classifier=-0.1153 logP_LM=-10.33
iter=005 step=09 accept=0.791 classifier=-0.0885 logP_LM=-9.87
iter=006 step=00 accept=0.348 classifier=-0.1207 logP_LM=-10.45
iter=006 step=09 accept=0.321 classifier=-0.0996 logP_LM=-10.26
iter=007 step=00 accept=0.956 classifier=-0.0942 logP_LM=-11.69
iter=007 step=09 accept=0.937 classifier=-0.0884 logP_LM=-11.20
iter=008 step=00 accept=0.733 classifier=-0.0814 logP_LM=-11.72
iter=008 step=09 accept=0.948 classifier=-0.0776 

KeyboardInterrupt: 